In [ ]:
# Databricks notebook source
# MAGIC %md
# MAGIC # 1️⃣ Bronze Layer: Ingestion & Normalisasi
# MAGIC **Arsitektur**: Medallion Pipeline  
# MAGIC **Dataset**: XAUUSD H1 OHLCV  
# MAGIC **Tujuan**: Membaca raw data CSV, standarisasi waktu, perhitungan `session_date` (17:00 NY boundary), dan `week_start`.

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, LongType, IntegerType

# ==============================================================================
# KONFIGURASI & PARAMETER
# ==============================================================================
# Sesuaikan path ini dengan lokasi file Anda di Unity Catalog Volume
RAW_FILE_PATH = "/Volumes/smt7_research/xauusd/raw_data/XAUUSD_H1.csv"
OUTPUT_TABLE = "smt7_research.xauusd.xauusd_liquidity_bronze"

SESSION_TZ = "America/New_York"
SESSION_SHIFT_HOURS = 7
START_DATE = "2016-01-01"

# Paksa sesi Spark ke UTC untuk determinisme from_utc_timestamp
spark.conf.set("spark.sql.session.timeZone", "UTC")

### 1. Schema & Ingestion

In [ ]:
# Schema untuk file CSV (Time, Open, High, Low, Close, Interval/Spread, Volume)
schema = StructType([
    StructField("Time", StringType(), True),
    StructField("Open", DoubleType(), True),
    StructField("High", DoubleType(), True),
    StructField("Low", DoubleType(), True),
    StructField("Close", DoubleType(), True),
    StructField("Spread", IntegerType(), True),
    StructField("Volume", LongType(), True)
])

# Membaca data CSV (dengan pemisah tab)
df_raw = spark.read.format("csv") \
    .option("header", "true") \
    .option("delimiter", "\t") \
    .schema(schema) \
    .load(RAW_FILE_PATH)

# Rename 'Time' jadi 'timestamp' dan ubah ke tipe timestamp
df_h1 = df_raw.withColumnRenamed("Time", "timestamp") \
              .filter(F.col("timestamp") >= START_DATE) \
              .withColumn("timestamp", F.to_timestamp("timestamp"))

# Pastikan lowercase nama kolom harga
df_h1 = df_h1.withColumnRenamed("Open", "open") \
             .withColumnRenamed("High", "high") \
             .withColumnRenamed("Low", "low") \
             .withColumnRenamed("Close", "close") \
             .withColumnRenamed("Volume", "volume")

### 2. Validasi Determinisme (Data Quality)

In [ ]:
# Cek duplikat timestamp yang akan merusak fungsi lead() / windowing
dup_h1 = df_h1.groupBy("timestamp").count().filter(F.col("count") > 1).count()
assert dup_h1 == 0, f"FATAL: Terdapat {dup_h1} duplikasi timestamp. lead() menjadi non-deterministik."

### 3. Transformasi: Session Date & Sesi Pasar (T13, T4)

In [ ]:
# T13: session_date (Hari berakhir pada 17:00 NY)
# from_utc_timestamp otomatis menangani DST. Geser +7 jam (24-17) agar berakhir di tengah malam logis.
ny_time = F.from_utc_timestamp(F.col("timestamp"), SESSION_TZ)
df_bronze = df_h1.withColumn(
    "session_date",
    F.to_date(ny_time + F.expr(f"INTERVAL {SESSION_SHIFT_HOURS} HOURS"))
)

# Cek hari perdagangan utuh (>= 12 candle)
cnt_days = df_bronze.groupBy("session_date").count()
assert cnt_days.filter(F.col("count") < 12).count() == 0, "FATAL: Ada hari perdagangan dengan <12 candle."
assert cnt_days.filter(F.dayofweek("session_date").isin(1, 7)).count() == 0, "FATAL: Ada sesi jatuh di Sabtu/Minggu (hari non-trading)."

# T4: Label Sesi Sadar Zona Waktu (ASIA, LONDON, NY, OFF_SESSION)
df_bronze = (df_bronze
    .withColumn("hour_ny",     F.hour(F.from_utc_timestamp("timestamp", "America/New_York")))
    .withColumn("hour_london", F.hour(F.from_utc_timestamp("timestamp", "Europe/London")))
    .withColumn("hour_tokyo",  F.hour(F.from_utc_timestamp("timestamp", "Asia/Tokyo")))
    .withColumn("session",
        F.when((F.col("hour_tokyo")  >= 9) & (F.col("hour_tokyo")  < 15), "ASIA")
         .when((F.col("hour_london") >= 8) & (F.col("hour_london") < 16), "LONDON")
         .when((F.col("hour_ny")     >= 8) & (F.col("hour_ny")     < 17), "NY")
         .otherwise("OFF_SESSION"))
)

### 4. Transformasi: Kunci Minggu / Week Start (T12)

In [ ]:
# T12: week_start (Tunggal & Monoton). Diturunkan dari session_date agar candle Minggu otomatis ke Senin.
df_bronze = df_bronze.withColumn("week_start", F.date_trunc("week", F.col("session_date")))

# Tripwire T12
bad_span = df_bronze.groupBy("week_start").agg(F.datediff(F.max("session_date"), F.min("session_date")).alias("span")).filter(F.col("span") > 7).count()
assert bad_span == 0, "FATAL: Ada kunci minggu merentang > 7 hari."
assert df_bronze.filter(F.dayofweek("week_start") != 2).count() == 0, "FATAL: week_start bukan hari Senin."

### 5. Penambahan Metadata & Simpan ke Delta Table

In [ ]:
# Tambahkan metadata ingestion
df_bronze = df_bronze.withColumn("ingested_at", F.current_timestamp())

# Simpan sebagai Delta Table
df_bronze.write.format("delta").mode("overwrite").saveAsTable(OUTPUT_TABLE)
print(f"Bronze layer saved to {OUTPUT_TABLE} dengan {df_bronze.count()} baris.")
display(df_bronze.limit(5))